# RQ1: Baseline Performance
**Research Question:** How effectively can baseline supervised learning models predict marketing campaign revenue on the Marketing and Product Performance Dataset?

**Task:** Regression — predicting `Revenue_Generated`

**Models:** Linear Regression, Decision Tree, k-NN

In [5]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os

os.makedirs('figures', exist_ok=True)
os.makedirs('tables', exist_ok=True)

In [6]:
DATA_PATH = '/kaggle/input/datasets/karansridhar/karan-ue-ml-assignment-dataset/marketing_and_product_performance.csv'
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head(3)

Shape: (10000, 17)
Columns: ['Campaign_ID', 'Product_ID', 'Budget', 'Clicks', 'Conversions', 'Revenue_Generated', 'ROI', 'Customer_ID', 'Subscription_Tier', 'Subscription_Length', 'Flash_Sale_ID', 'Discount_Level', 'Units_Sold', 'Bundle_ID', 'Bundle_Price', 'Customer_Satisfaction_Post_Refund', 'Common_Keywords']


,Campaign_ID,Product_ID,Budget,Clicks,Conversions,Revenue_Generated,ROI,Customer_ID,Subscription_Tier,Subscription_Length,Flash_Sale_ID,Discount_Level,Units_Sold,Bundle_ID,Bundle_Price,Customer_Satisfaction_Post_Refund,Common_Keywords
0,CMP_RLSDVN,PROD_HBJFA3,41770.45,4946,73,15520.09,1.94,CUST_1K7G39,Premium,4,FLASH_1VFK5K,43,34,BNDL_29U6W5,433.80,4,Affordable
1,CMP_JHHUE9,PROD_OE8YNJ,29900.93,570,510,30866.17,0.76,CUST_0DWS6F,Premium,4,FLASH_1M6COK,28,97,BNDL_ULV60J,289.29,2,Innovative
2,CMP_6SBOWN,PROD_4V8A08,22367.45,3546,265,32585.62,1.41,CUST_BR2GST,Basic,9,FLASH_J4PEON,51,160,BNDL_0HY0EF,462.87,4,Affordable


In [7]:
TARGET = 'Revenue_Generated'

id_cols = [c for c in df.columns if 'ID' in c or 'id' in c.lower()]
df_model = df.drop(columns=id_cols, errors='ignore').dropna(subset=[TARGET])

cat_cols = df_model.select_dtypes(include='object').columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col].fillna('Unknown').astype(str))

df_model = df_model.fillna(df_model.median(numeric_only=True))

X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (8000, 11), Test: (2000, 11)


In [8]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree':     DecisionTreeRegressor(max_depth=6, random_state=42),
    'k-NN':              KNeighborsRegressor(n_neighbors=7)
}

results = []
for name, model in models.items():
    X_tr = X_train_sc if name in ['Linear Regression', 'k-NN'] else X_train
    X_te = X_test_sc  if name in ['Linear Regression', 'k-NN'] else X_test
    model.fit(X_tr, y_train)
    preds = model.predict(X_te)
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    results.append({'Model': name, 'MAE': round(mae,4), 'RMSE': round(rmse,4), 'R²': round(r2,4)})
    print(f'{name:22s} | MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}')

df_results = pd.DataFrame(results)
df_results.to_csv('tables/RQ1_Table1_Baseline_Performance.csv', index=False)
print('\nTable saved.')
print(df_results)

Linear Regression      | MAE=24745.4005  RMSE=28590.8029  R²=-0.0025
Decision Tree          | MAE=24972.8386  RMSE=29014.9427  R²=-0.0324
k-NN                   | MAE=25728.5757  RMSE=30429.1341  R²=-0.1355

Table saved.
               Model         MAE        RMSE      R²
0  Linear Regression  24745.4005  28590.8029 -0.0025
1      Decision Tree  24972.8386  29014.9427 -0.0324
2               k-NN  25728.5757  30429.1341 -0.1355


In [9]:
# Figure 1 — Dual-axis chart: MAE/RMSE on left axis, R² on right axis
# (They cannot share one axis — MAE/RMSE are in thousands, R² is 0–1)

x = np.arange(len(df_results))
width = 0.25

fig, ax1 = plt.subplots(figsize=(11, 6))
ax2 = ax1.twinx()  # second y-axis for R²

b1 = ax1.bar(x - width/2,   df_results['MAE'],  width, label='MAE',  color='#2196F3', edgecolor='white')
b2 = ax1.bar(x + width/2,   df_results['RMSE'], width, label='RMSE', color='#FF5722', edgecolor='white')
b3 = ax2.bar(x + width*1.5, df_results['R²'],   width, label='R²',   color='#4CAF50', edgecolor='white')

ax1.set_xticks(x + width/2)
ax1.set_xticklabels(df_results['Model'], fontsize=12)
ax1.set_ylabel('MAE / RMSE', fontsize=12)
ax2.set_ylabel('R²  (0 – 1)', fontsize=12, color='#2E7D32')
ax2.set_ylim(0, 1.2)
ax2.tick_params(axis='y', labelcolor='#2E7D32')

ax1.set_title('Figure 1: Baseline Model Performance\n(Marketing and Product Performance Dataset)',
              fontsize=13, fontweight='bold')

handles = [b1, b2, b3]
labels  = [h.get_label() for h in handles]
ax1.legend(handles, labels, fontsize=11, loc='upper left')
ax1.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/RQ1_Figure1_Baseline_Performance.pdf', bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.
